In [78]:
from src.utils import getSparkSession
spark = getSparkSession()

In [131]:
# Preview de los datos
df_raw = spark.read.table("local.db.tipos_cambio_raw")
df_raw.show(3, truncate=False)
df_enriquecidos = spark.read.table("local.db.tipos_cambio_enriquecidos")
df_enriquecidos.show(3, truncate=False)
df_fact = spark.read.table("local.db.fact_exchange_rates")
df_fact.show(5, truncate=False)

+----------+----+------+-------+------+------+--------------------------+
|date      |base|MXN   |EUR    |AUD   |CAD   |fecha_carga               |
+----------+----+------+-------+------+------+--------------------------+
|2023-12-29|USD |16.944|0.90498|1.4718|1.3251|2026-05-19 17:13:48.162078|
|2024-01-02|USD |17.058|0.91274|1.4738|1.3294|2026-05-19 17:13:48.162078|
|2024-01-03|USD |17.097|0.91583|1.4869|1.3347|2026-05-19 17:13:48.162078|
+----------+----+------+-------+------+------+--------------------------+
only showing top 3 rows

+-------------+-------------+---------------+-------------+------------------+-------------+--------------------------+--------------+--------------+--------------------------+-----------+---------------+---------+--------------+--------------------------+--------------------------+
|exchange_date|base_currency|target_currency|exchange_rate|is_originally_null|moving_avg_7d|transformed_at            |moving_avg_30d|rate_yesterday|variacion_pj_diaria_exra

In [132]:
# Revisión de esquema.
print("Raw:")
df_raw.printSchema()
print("Silver:")
df_enriquecidos.printSchema()
print("Gold:")
df_fact.printSchema()

Raw:
root
 |-- date: date (nullable = true)
 |-- base: string (nullable = true)
 |-- MXN: double (nullable = true)
 |-- EUR: double (nullable = true)
 |-- AUD: double (nullable = true)
 |-- CAD: double (nullable = true)
 |-- fecha_carga: timestamp (nullable = true)

Silver:
root
 |-- exchange_date: date (nullable = true)
 |-- base_currency: string (nullable = true)
 |-- target_currency: string (nullable = true)
 |-- exchange_rate: double (nullable = true)
 |-- is_originally_null: integer (nullable = true)
 |-- moving_avg_7d: double (nullable = true)
 |-- transformed_at: timestamp (nullable = true)
 |-- moving_avg_30d: double (nullable = true)
 |-- rate_yesterday: double (nullable = true)
 |-- variacion_pj_diaria_exrate: double (nullable = true)
 |-- dev_std_30d: double (nullable = true)
 |-- racha_rellenado: long (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- operation_type: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- updated_at

In [102]:
df_silver.columns


['exchange_date',
 'base_currency',
 'target_currency',
 'exchange_rate',
 'is_originally_null',
 'moving_avg_7d',
 'transformed_at',
 'moving_avg_30d',
 'rate_yesterday',
 'variacion_pj_diaria_exrate',
 'dev_std_30d',
 'racha_rellenado',
 'is_active',
 'operation_type',
 'ingestion_timestamp',
 'updated_at']

In [133]:
# Revision completitud de los datos por fines de semana o dias festivos.
df_silver_conteo_mes = df_enriquecidos.where(F.col("target_currency") == "MXN").groupBy(F.date_format(F.col("exchange_date"), "yyyy-MM").alias("year_month")).count().orderBy(F.col("year_month").desc())
df_silver_conteo_mes.show(20, truncate=False)

+----------+-----+
|year_month|count|
+----------+-----+
|2026-05   |19   |
|2026-04   |30   |
|2026-03   |31   |
|2026-02   |28   |
|2026-01   |31   |
|2025-12   |31   |
|2025-11   |30   |
|2025-10   |31   |
|2025-09   |30   |
|2025-08   |31   |
|2025-07   |31   |
|2025-06   |30   |
|2025-05   |31   |
|2025-04   |30   |
|2025-03   |31   |
|2025-02   |28   |
|2025-01   |31   |
|2024-12   |31   |
|2024-11   |30   |
|2024-10   |31   |
+----------+-----+
only showing top 20 rows



In [104]:
df_registros_rellenados = df_enriquecidos.filter(F.col("is_originally_null") == 1)
df_registros_rellenados.show(10, truncate=False)

+-------------+-------------+---------------+-------------+------------------+-------------+--------------------------+--------------+--------------+--------------------------+-----------+---------------+---------+--------------+--------------------------+--------------------------+
|exchange_date|base_currency|target_currency|exchange_rate|is_originally_null|moving_avg_7d|transformed_at            |moving_avg_30d|rate_yesterday|variacion_pj_diaria_exrate|dev_std_30d|racha_rellenado|is_active|operation_type|ingestion_timestamp       |updated_at                |
+-------------+-------------+---------------+-------------+------------------+-------------+--------------------------+--------------+--------------+--------------------------+-----------+---------------+---------+--------------+--------------------------+--------------------------+
|2024-01-01   |USD          |AUD            |1.4718       |1                 |1.4718       |2026-05-19 06:55:03.656052|1.4718        |1.4718        

In [111]:
# Preview de operation_type, is_active
# spark.conf.set("spark.hadoop.fs.file.impl.disable.cache", "true")
df_ope = df_enriquecidos.groupBy(
    F.col("operation_type"), 
    F.col("is_active")) \
    .agg(
        F.count("*").alias("Registros")
    )
df_ope.show()

+--------------+---------+---------+
|operation_type|is_active|Registros|
+--------------+---------+---------+
|        INSERT|     true|     3492|
+--------------+---------+---------+



In [130]:
df_enrachados = df_enriquecidos.filter( 
    ((F.col("racha_rellenado") >= 1) & (F.col("racha_rellenado") <= 3)) &  
    (F.col("target_currency") == "MXN")).orderBy(
        F.col("exchange_date").asc(),
        F.col("racha_rellenado").asc()
    )
df_enrachados.show(5, truncate=False)

+-------------+-------------+---------------+-------------+------------------+-------------+--------------------------+--------------+--------------+--------------------------+-----------+---------------+---------+--------------+--------------------------+--------------------------+
|exchange_date|base_currency|target_currency|exchange_rate|is_originally_null|moving_avg_7d|transformed_at            |moving_avg_30d|rate_yesterday|variacion_pj_diaria_exrate|dev_std_30d|racha_rellenado|is_active|operation_type|ingestion_timestamp       |updated_at                |
+-------------+-------------+---------------+-------------+------------------+-------------+--------------------------+--------------+--------------+--------------------------+-----------+---------------+---------+--------------+--------------------------+--------------------------+
|2023-12-30   |USD          |MXN            |16.944       |1                 |16.944       |2026-05-19 06:55:03.656052|16.944        |16.944        